# Decision Tree Classification - Titanic Survival Prediction

## Objective

Build a complete Decision Tree Classification pipeline to predict passenger survival using a small Titanic dataset.

This notebook demonstrates:

- Loading data
- Handling missing values
- One-Hot Encoding
- Train-Test Split
- Training Decision Tree
- Model Evaluation
- Feature Importance Analysis

# Problem Statement

An insurance analytics company wants to build a machine learning model to predict whether a passenger survived the Titanic disaster.

The dataset contains passenger details including:

- Passenger Class
- Gender
- Age
- Family Information
- Fare
- Port of Embarkation

The objective is to:

- Clean the data
- Encode categorical variables
- Train a Decision Tree Classifier
- Evaluate model accuracy
- Identify the most important feature influencing survival

In [1]:
import pandas as pd
from io import StringIO

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
#Load Dataset 
sample_csv = """pclass,sex,age,sibsp,parch,fare,embarked,survived
1,female,29,0,0,211.34,S,1
1,male,42,0,0,51.86,S,0
2,female,,1,0,21.00,S,1
2,male,35,0,0,13.00,S,0
3,female,22,1,0,7.25,S,1
3,male,32,0,0,8.05,S,0
1,female,38,1,0,71.28,C,1
3,male,19,0,0,8.66,Q,0
2,male,28,0,0,13.00,S,0
3,female,4,1,1,16.70,,1
1,male,54,0,1,51.86,S,0
3,female,14,0,0,7.85,S,1
"""

df = pd.read_csv(StringIO(sample_csv))

print(df)

    pclass     sex   age  sibsp  parch    fare embarked  survived
0        1  female  29.0      0      0  211.34        S         1
1        1    male  42.0      0      0   51.86        S         0
2        2  female   NaN      1      0   21.00        S         1
3        2    male  35.0      0      0   13.00        S         0
4        3  female  22.0      1      0    7.25        S         1
5        3    male  32.0      0      0    8.05        S         0
6        1  female  38.0      1      0   71.28        C         1
7        3    male  19.0      0      0    8.66        Q         0
8        2    male  28.0      0      0   13.00        S         0
9        3  female   4.0      1      1   16.70      NaN         1
10       1    male  54.0      0      1   51.86        S         0
11       3  female  14.0      0      0    7.85        S         1


In [3]:
# Handle Missing Values

# - Fill missing **Age** values using the **Median**.
# - Fill missing **Embarked** values using the **Mode**.

df["age"] = df["age"].fillna(df["age"].median())

df["embarked"] = df["embarked"].fillna(
    df["embarked"].mode()[0]
)

print(df)

    pclass     sex   age  sibsp  parch    fare embarked  survived
0        1  female  29.0      0      0  211.34        S         1
1        1    male  42.0      0      0   51.86        S         0
2        2  female  29.0      1      0   21.00        S         1
3        2    male  35.0      0      0   13.00        S         0
4        3  female  22.0      1      0    7.25        S         1
5        3    male  32.0      0      0    8.05        S         0
6        1  female  38.0      1      0   71.28        C         1
7        3    male  19.0      0      0    8.66        Q         0
8        2    male  28.0      0      0   13.00        S         0
9        3  female   4.0      1      1   16.70        S         1
10       1    male  54.0      0      1   51.86        S         0
11       3  female  14.0      0      0    7.85        S         1


# One-Hot Encoding

Convert categorical columns:

- sex
- embarked

using:

```python
pd.get_dummies(drop_first=True)
```

This converts text values into numerical values that the model can understand.

In [4]:
df = pd.get_dummies(
    df,
    columns=["sex", "embarked"],
    drop_first=True
)

print(df.head())

   pclass   age  sibsp  parch    fare  survived  sex_male  embarked_Q  \
0       1  29.0      0      0  211.34         1     False       False   
1       1  42.0      0      0   51.86         0      True       False   
2       2  29.0      1      0   21.00         1     False       False   
3       2  35.0      0      0   13.00         0      True       False   
4       3  22.0      1      0    7.25         1     False       False   

   embarked_S  
0        True  
1        True  
2        True  
3        True  
4        True  


In [5]:
# Define Features and Target

# Features (X):

# All columns except **survived**

# Target (y):

# **survived**
X = df.drop("survived", axis=1)

y = df["survived"]

# Train-Test Split

Split the dataset into:

- 75% Training Data
- 25% Testing Data

using:

- test_size = 0.25
- random_state = 42
- stratify = y

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [7]:
# Train Decision Tree Classifier

# Model Parameters

# - criterion = gini
# - max_depth = 3
# - min_samples_leaf = 1
# - random_state = 42

model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    min_samples_leaf=1,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

DecisionTreeClassifier(max_depth=3, random_state=42)

In [8]:
# Predict Test Data

y_pred = model.predict(X_test)


In [9]:
#Model Accuracy
accuracy = accuracy_score(
    y_test,
    y_pred
)

print(f"Test Accuracy : {accuracy:.4f}")

Test Accuracy : 1.0000


# Feature Importance

Decision Trees calculate the importance of every feature.

The feature with the highest importance contributes the most toward prediction.

In [10]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

print("\nMost Important Feature:")

print(feature_importance.iloc[0])

      Feature  Importance
5    sex_male         1.0
0      pclass         0.0
2       sibsp         0.0
1         age         0.0
3       parch         0.0
4        fare         0.0
6  embarked_Q         0.0
7  embarked_S         0.0

Most Important Feature:
Feature       sex_male
Importance         1.0
Name: 5, dtype: object


# Conclusion

Workflow

Dataset

↓

Handle Missing Values

↓

One-Hot Encoding

↓

Train-Test Split

↓

Decision Tree Training

↓

Prediction

↓

Accuracy Evaluation

↓

Feature Importance

The model successfully predicts passenger survival and identifies the most influential feature affecting survival.